# RSA

## 0. Setup

In [26]:
import cryptography
from utils import *
print("cryptography:", cryptography.__version__)

cryptography: 50.0.0


# 1. RSA Key Generation — High Level

Generiamo una coppia di chiavi RSA usando `cryptography`.

$$n=pq$$


In [27]:
from cryptography.hazmat.primitives.asymmetric import rsa

private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

print("RSA key generated")
print("Key size:", private_key.key_size)


RSA key generated
Key size: 2048


## Parametri RSA

L'API nasconde i dettagli matematici, ma possiamo recuperarli.

In [30]:
numbers = private_key.private_numbers()

p = numbers.p
q = numbers.q
d = numbers.d

public_numbers = numbers.public_numbers
n = public_numbers.n
e = public_numbers.e


print("=== PUBLIC KEY ===")
print_openssl_hex("n", n)
print()
print(f"e =  {e} (0x{e:x})")

print("\n=== PRIVATE KEY ===")
print_openssl_hex("d", d)
print()
print_openssl_hex("p", p)
print()
print_openssl_hex("q", q)

=== PUBLIC KEY ===
n =  00:bc:bd:00:a7:e7:c7:5f:1b:8e:84:3a:71:97:fc:71
     f2:ff:40:f1:00:c4:d8:f0:7a:03:6a:49:dc:91:f2:ea
     8d:17:6b:2d:23:a9:94:a7:91:9d:19:cd:21:b9:9d:31
     9f:4c:0e:0d:a6:25:09:97:25:6f:e7:54:8d:32:e7:9e
     f9:47:89:c2:bd:a6:58:3a:a0:2c:e8:a5:51:42:93:46
     45:d3:3b:22:fb:01:6d:c7:dc:60:02:d9:ce:dd:b9:8d
     12:7a:cb:35:67:78:70:05:27:66:ce:7b:e9:61:8d:e0
     dd:fc:d8:f0:04:94:61:a4:c7:e2:75:fe:dd:aa:b1:a6
     dc:ab:a5:1a:4d:07:b8:48:a9:ee:c6:1d:85:d7:e6:11
     07:7c:e1:7e:26:64:6d:70:17:b4:c6:a7:0c:31:fd:4a
     5d:f5:f1:8d:2a:77:9a:51:d9:26:a1:64:83:d2:db:c1
     ea:b0:03:13:b5:94:a0:1a:b7:68:12:f4:0d:91:4f:4d
     2f:96:d0:8f:1e:11:99:c6:6d:86:cb:88:9c:d0:24:b8
     e8:54:25:f4:bc:9e:c5:fa:ff:ba:7f:1b:14:b3:41:c0
     a0:2a:3e:42:a8:6d:f6:89:6e:4e:b9:8c:96:04:84:90
     98:ea:08:ec:ee:24:78:45:1d:95:6f:e2:02:c7:88:ef
     8b

e =  65537 (0x10001)

=== PRIVATE KEY ===
d =  52:4b:38:bf:7d:5c:dd:97:d6:de:c9:da:e0:d1:c6:5f
     95:33:9c:41:e1:12:d4:5b:

# 2. RSA come Key Transport / KEM-like

RSA non espone una KEM API moderna come ML-KEM.

Possiamo però usare RSA-OAEP per un **key transport**:

```text
Alice                                      Bob
  │                                         │
  │              public key                 │
  │◄────────────────────────────────────────│
  │                                         │
  │ random secret                            │
  │                                         │
  │──── RSA-OAEP(secret) ──────────────────►│
  │                                         │
  │                                  RSA-OAEP decrypt
  │                                         │
  │                                  shared secret
```

Questo è utile come ponte concettuale verso:

`generate → encaps → decaps`


In [ ]:
import os

secret = os.urandom(32)
print("Secret:", secret.hex())
print("Length:", len(secret), "bytes")


### Encapsulation / Key Transport

In [ ]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding

ciphertext = public_key.encrypt(
    secret,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Ciphertext length:", len(ciphertext), "bytes")


### Decapsulation / Recovery

In [ ]:
recovered_secret = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print("Match:", recovered_secret == secret)
assert recovered_secret == secret


> **Nota:** RSA-OAEP è encryption/key transport, non una KEM standard come ML-KEM. Lo usiamo qui per mostrare l'analogia API.

# 3. RSA come Digital Signature

Per le firme usiamo **RSA-PSS**.

In [ ]:
message = b"RSA hands-on"

signature = private_key.sign(
    message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print("Signature length:", len(signature), "bytes")


In [ ]:
public_key.verify(
    signature,
    message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print("Signature valid")


### Modifichiamo il messaggio

In [ ]:
tampered_message = b"RSA hands-on - modified"

try:
    public_key.verify(
        signature,
        tampered_message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature valid")
except Exception:
    print("Signature INVALID")


# 4. Dall'API alla primitive RSA

Le API nascondono una primitive fondamentale:

$$y=x^e\bmod n$$

oppure:

$$y=x^d\bmod n$$

Come calcoliamo questa operazione in maniera efficiente?

In [ ]:
def modular_pow_naive(x, e, n):
    result = 1
    for _ in range(e):
        result = (result * x) % n
    return result

print(modular_pow_naive(3, 13, 55))
print(pow(3, 13, 55))


La versione naïve richiede circa:

$$O(e)$$

moltiplicazioni.

Per RSA usiamo la rappresentazione binaria dell'esponente.

In [ ]:
e = 13
print("Decimal:", e)
print("Binary :", bin(e))


# 5. Square-and-Multiply

$$13=(1101)_2=8+4+1$$

Quindi possiamo costruire:

$$x^2, x^4, x^8, ...$$

e ottenere $x^{13}$ con un numero di operazioni dell'ordine di:

$$O(\log e)$$

In [ ]:
def modular_pow(x, e, n):
    result = 1
    x %= n

    while e > 0:
        if e & 1:
            result = (result * x) % n

        x = (x * x) % n
        e >>= 1

    return result


x, e, n = 3, 13, 55

print("Our implementation:", modular_pow(x, e, n))
print("Python pow():      ", pow(x, e, n))
assert modular_pow(x, e, n) == pow(x, e, n)


### Vediamo i singoli passi

In [ ]:
def show_square_multiply(x, e, n):
    result = 1
    x %= n
    step = 0

    while e > 0:
        print(f"step {step}: bit={e & 1}, result={result}, x={x}, e={e}")

        if e & 1:
            result = (result * x) % n

        x = (x * x) % n
        e >>= 1
        step += 1

    print("final result =", result)
    return result

show_square_multiply(3, 13, 55)


# 6. Modular Inverse — Extended Euclidean Algorithm

Durante RSA dobbiamo trovare:

$$d=e^{-1}\bmod\varphi(n)$$

cioè:

$$ed\equiv1\pmod{\varphi(n)}$$

L'algoritmo di Euclide esteso permette di trovare $x,y$ tali che:

$$ax+by=\gcd(a,b)$$

In [ ]:
def extended_gcd(a, b):
    if b == 0:
        return a, 1, 0

    g, x1, y1 = extended_gcd(b, a % b)

    x = y1
    y = x1 - (a // b) * y1

    return g, x, y


In [ ]:
a, b = 7, 160

g, x, y = extended_gcd(a, b)

print("gcd =", g)
print("x   =", x)
print("y   =", y)
print("a*x + b*y =", a*x + b*y)


In [ ]:
a, modulus = 7, 160

g, x, _ = extended_gcd(a, modulus)
assert g == 1

inverse = x % modulus

print("Inverse:", inverse)
print("Check:", (a * inverse) % modulus)


# 7. RSA Toy Example

Usiamo numeri piccoli per vedere tutti i passaggi:

$$p=11,\quad q=17$$

$$n=pq$$

$$\varphi(n)=(p-1)(q-1)$$

Scegliamo $e=7$ e calcoliamo $d$.

In [ ]:
from math import gcd

p, q = 11, 17
n = p * q
phi = (p - 1) * (q - 1)

e = 7
assert gcd(e, phi) == 1

d = pow(e, -1, phi)

print("p   =", p)
print("q   =", q)
print("n   =", n)
print("phi =", phi)
print("e   =", e)
print("d   =", d)
print("e*d mod phi =", (e * d) % phi)


### Encryption / Decryption

In [ ]:
m = 42

c = modular_pow(m, e, n)
m_recovered = modular_pow(c, d, n)

print("m  =", m)
print("c  =", c)
print("m' =", m_recovered)

assert m_recovered == m


# 8. CRT — Chinese Remainder Theorem

Per accelerare la decifratura RSA possiamo sfruttare:

$$n=pq$$

Calcoliamo:

$$d_p=d\bmod(p-1)$$

$$d_q=d\bmod(q-1)$$

poi:

$$m_1=c^{d_p}\bmod p$$

$$m_2=c^{d_q}\bmod q$$

e ricostruiamo $m$ con CRT.

In [ ]:
def rsa_crt_decrypt(c, p, q, d):
    dp = d % (p - 1)
    dq = d % (q - 1)

    m1 = pow(c, dp, p)
    m2 = pow(c, dq, q)

    q_inv = pow(q, -1, p)

    h = (q_inv * (m1 - m2)) % p
    return m2 + h * q


In [ ]:
m_direct = pow(c, d, n)
m_crt = rsa_crt_decrypt(c, p, q, d)

print("Direct:", m_direct)
print("CRT   :", m_crt)

assert m_direct == m_crt


### Schema CRT

```text
              RSA decryption
                    │
             c^d mod n
                    │
              ┌─────┴─────┐
              ▼           ▼
          c^dp mod p   c^dq mod q
              │           │
              └─────┬─────┘
                    ▼
                   CRT
                    │
                    ▼
                    m
```

Lavoriamo con moduli più piccoli e poi ricombiniamo il risultato.

# 9. Benchmark concettuale

Naïve:

$$O(e)$$

Square-and-multiply:

$$O(\log e)$$



In [ ]:
def count_naive(e):
    return e

def count_square_multiply(e):
    operations = 0

    while e > 0:
        if e & 1:
            operations += 1
        e >>= 1
        if e > 0:
            operations += 1

    return operations

for e in [13, 65537, 2**16, 2**32]:
    print(
        f"e={e:<12} "
        f"naive≈{count_naive(e):<15} "
        f"square-multiply≈{count_square_multiply(e)}"
    )


# 10. Dal codice didattico all'implementazione reale

```text
High-level API
      │
      ├── RSA-OAEP
      │     └── key transport / KEM-like
      │
      └── RSA-PSS
            └── digital signature
                  │
                  ▼
              RSA primitive
                  │
          ┌───────┼────────┐
          ▼       ▼        ▼
       modular  modular    CRT
       inverse  exponent.
          │       │
          ▼       ▼
      Euclid   Square-and-Multiply
```

Le implementazioni reali aggiungono anche:

- big-integer arithmetic;
- moltiplicazioni efficienti per interi grandi;
- CRT;
- constant-time implementations;
- blinding;
- padding e encoding sicuri;
- protezioni contro side-channel.

Le funzioni implementate nel notebook sono **didattiche**, non sostituiscono
`cryptography`/OpenSSL.

# 11. Collegamento con le primitive post-quantum

Lo stesso schema verrà riutilizzato con `liboqs-python`.

### RSA

```text
generate
   │
   ├── OAEP → encrypt/decrypt
   │
   └── PSS  → sign/verify
```

### ML-KEM

```text
generate_keypair
        │
        ├── encaps
        │      └── ciphertext + shared secret
        │
        └── decaps
               └── shared secret
```

### ML-DSA / SLH-DSA

```text
generate_keypair
        │
        ├── sign
        │
        └── verify
```

La domanda successiva sarà:

> Quali primitive matematiche e quali ottimizzazioni ci sono sotto
> `generate`, `encaps`, `decaps`, `sign` e `verify` nelle primitive
> post-quantum?

# Takeaway

$$
\boxed{
\text{API}
\rightarrow
\text{primitive}
\rightarrow
\text{algoritmo efficiente}
\rightarrow
\text{implementazione reale}
}
$$

Per RSA abbiamo visto:

- RSA-OAEP → key transport
- RSA-PSS → digital signature
- modular exponentiation
- square-and-multiply
- modular inverse / Extended Euclid
- CRT

Questo schema può essere riutilizzato per le primitive post-quantum.